In [ ]:
!pip install transformers datasets torch gradio scikit-learn -q

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("davanstrien/WELFake")
df = dataset["train"].to_pandas()
df = df.dropna(subset=["title", "text"])
df["input"] = (df["title"] + " " + df["text"]).str[:2000]
df = df.reset_index(drop=True)
print(f"Total samples: {len(df)}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-290868f0a36350(…):   0%|          | 0.00/152M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/72134 [00:00<?, ? examples/s]

Total samples: 71537


In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("davanstrien/WELFake")
df = dataset["train"].to_pandas()
df = df.dropna(subset=["title", "text"])
df["input"] = (df["title"] + " " + df["text"]).str[:2000]
df = df.reset_index(drop=True)
print(f"Total samples: {len(df)}")

Total samples: 71537


In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import torch

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
train_df, val_df  = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df["label"])

# Use small subset
train_df = train_df.sample(10000, random_state=42).reset_index(drop=True)
val_df   = val_df.sample(1000, random_state=42).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(texts):
    return tokenizer(list(texts), truncation=True, padding=True, max_length=512, return_tensors="pt")

train_enc = tokenize(train_df["input"])
val_enc   = tokenize(val_df["input"])
print("Tokenization done ")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenization done 


In [ ]:
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification

class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = FakeNewsDataset(train_enc, train_df["label"])
val_dataset   = FakeNewsDataset(val_enc,   val_df["label"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model  = model.to(device)
print(f"Device: {device} ")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda 


In [ ]:
from torch.optim import AdamW
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
optimizer    = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            preds = model(input_ids=input_ids, attention_mask=attention_mask).logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {correct/total*100:.2f}%")

print("\nTraining done ")

Epoch 1/3 | Loss: 0.1225 | Val Acc: 97.60%
Epoch 2/3 | Loss: 0.0233 | Val Acc: 99.20%
Epoch 3/3 | Loss: 0.0101 | Val Acc: 99.60%

Training done 


In [ ]:
from sklearn.metrics import classification_report

test_enc  = tokenize(test_df["input"])
test_dataset = FakeNewsDataset(test_enc, test_df["label"])
test_loader  = DataLoader(test_dataset, batch_size=32)

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)
        preds = model(input_ids=input_ids, attention_mask=attention_mask).logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=["Real", "Fake"]))

              precision    recall  f1-score   support

        Real       1.00      0.98      0.99      7006
        Fake       0.98      1.00      0.99      7302

    accuracy                           0.99     14308
   macro avg       0.99      0.99      0.99     14308
weighted avg       0.99      0.99      0.99     14308



In [ ]:
import gradio as gr

def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs   = outputs.logits.softmax(dim=-1)
        pred    = probs.argmax().item()
        confidence = probs.max().item() * 100

    label = "🔴 FAKE NEWS" if pred == 1 else "🟢 REAL NEWS"
    return f"{label}\nConfidence: {confidence:.2f}%"

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=5, placeholder="Paste a news headline or article here..."),
    outputs=gr.Textbox(label="Prediction"),
    title="Fake News Detector",
    description="Fine-tuned DistilBERT model trained on 10,000 news articles."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3c064967719d934410.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from huggingface_hub import login
login()

HF_USERNAME = "Musadiq7860"
MODEL_NAME  = f"{HF_USERNAME}/fake-news-detector"

model.push_to_hub(MODEL_NAME)
tokenizer.push_to_hub(MODEL_NAME)

print(f"Model pushed ✅")
print(f"https://huggingface.co/{MODEL_NAME}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...yeshawy/model.safetensors:   0%|          |  575kB /  268MB            

README.md: 0.00B [00:00, ?B/s]

Model pushed ✅
https://huggingface.co/Musadiq7860/fake-news-detector
